<a href="https://colab.research.google.com/github/skshahid0786/0x44/blob/main/SmolLM_135M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:

# 1. Setup
!pip install -q -U trl transformers accelerate datasets gradio
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 2. Data
dataset = load_dataset("nvidia/HelpSteer", split="train[:500]")
dataset = dataset.rename_column("response", "completion")

# 3. Train
trainer = SFTTrainer(
    model="HuggingFaceTB/SmolLM2-135M",
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="prompt",
        max_length=512,
        output_dir="jarvis_brain", # Simple name
        per_device_train_batch_size=2,
        num_train_epochs=1,
        learning_rate=5e-5,
        report_to="none"
    ),
)

trainer.train()
trainer.save_model("jarvis_brain")
print("Jarvis brain is finished and saved in 'jarvis_brain'!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.4 MB/s eta 0:00:00


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss
10,2.635109
20,1.158872
30,1.304749
40,1.308208
50,1.694321
60,1.473783
70,1.801011
80,1.584613
90,2.337057
100,1.788013


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Jarvis brain is finished and saved in 'jarvis_brain'!


## Local Inference on GPU
Model page: https://huggingface.co/HuggingFaceTB/SmolLM-135M

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/HuggingFaceTB/SmolLM-135M)
            and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [16]:

import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Use the exact same name as Step 1
model_path = "jarvis_brain"

print("Loading the brain...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def chat_with_jarvis(message, history):
    system_rules = "You are JARVIS, Shahid's AI. Be professional and brief."
    prompt = f"System: {system_rules}\nUser: {message}\nAssistant:"

    output = pipe(prompt, max_new_tokens=60, do_sample=True, temperature=0.3, repetition_penalty=1.4)
    response = output[0]['generated_text'].split("Assistant:")[-1].strip()
    return response.split("User:")[0].strip()

demo = gr.ChatInterface(fn=chat_with_jarvis, title="J.A.R.V.I.S. PRO")
demo.launch(share=True)

Loading the brain...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ca097e589ad9bd6c90.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
